# FMP Evidence-Based Daily-Adjusted Fundamentals

This notebook tests whether daily-adjusted valuation features built from FMP historical data add signal versus static/as-of quarterly fundamentals.

Rules for this analysis:

- Use only data already stored in Quant Warehouse.
- Use historical data only: daily prices, daily historical market cap, quarterly income/balance/cash/ratios/metrics.
- Do not use snapshot endpoints such as share statistics.
- Do not call the FMP API directly.
- Do not use `historical_enterprise_value`; daily enterprise value is reconstructed from daily market cap plus last-known debt minus last-known cash.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable
from time import perf_counter
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from quant_warehouse import Warehouse

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

PROVIDER = "fmp"
MAG7 = ("AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "TSLA")
# Full MAG7 historical run. Use a smaller ANALYSIS_SYMBOLS/START_DATE smoke config when changing logic.
ANALYSIS_SYMBOLS = MAG7
START_DATE = "2018-01-01"
END_DATE = None
FILING_LAG_DAYS = 45
HORIZONS = (20, 60, 120)
MIN_OBS = 120

FULL_RUN_BASELINE_SYMBOLS = MAG7
FULL_RUN_BASELINE_START_DATE = "2018-01-01"

wh = Warehouse()
run_timings: dict[str, float] = {}


## Feature Families

The notebook compares four groups:

1. **Daily market-cap adjusted valuation**: last-known fundamentals divided by daily historical market cap, or the inverse.
2. **Daily reconstructed EV valuation**: daily EV equals daily market cap plus last-known total debt minus last-known cash.
3. **Static vendor valuation benchmarks**: FMP ratios/metrics forward-filled after the filing lag.
4. **As-of quality controls**: margins, returns, leverage, growth, and balance-sheet quality that should not be daily repriced.

The scoring convention normalizes direction before measuring predictive performance: higher score should imply higher expected forward return.


In [2]:
@dataclass(frozen=True)
class FeatureSpec:
    feature: str
    family: str
    variant: str
    expected_direction: str
    source: str


def _date_indexed(frame: pd.DataFrame) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    out = frame.copy()
    if not isinstance(out.index, pd.DatetimeIndex):
        out.index = pd.to_datetime(out.index, errors="coerce")
    out = out.loc[out.index.notna()].sort_index()
    out.index = pd.DatetimeIndex(out.index).normalize()
    return out


def _slice(frame: pd.DataFrame, start: str | None = None, end: str | None = None) -> pd.DataFrame:
    out = _date_indexed(frame)
    if out.empty:
        return out
    if start:
        out = out.loc[out.index >= pd.Timestamp(start)]
    if end:
        out = out.loc[out.index <= pd.Timestamp(end)]
    return out


def _dedupe_asof(values: pd.Series | pd.DataFrame, *, lag_days: int = FILING_LAG_DAYS):
    out = values.copy()
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index, errors="coerce")).normalize() + pd.Timedelta(days=int(lag_days))
    out = out.loc[out.index.notna()].sort_index()
    return out.loc[~out.index.duplicated(keep="last")]


def _asof_to_daily(values: pd.Series | pd.DataFrame, daily_index: pd.DatetimeIndex, *, lag_days: int = FILING_LAG_DAYS):
    sparse = _dedupe_asof(values, lag_days=lag_days)
    return sparse.reindex(daily_index, method="ffill")


def _numeric(frame: pd.DataFrame, column: str) -> pd.Series:
    if frame.empty or column not in frame.columns:
        return pd.Series(dtype="float64")
    return pd.to_numeric(frame[column], errors="coerce").replace([np.inf, -np.inf], np.nan)


def _safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    out = numerator / denominator.replace(0, np.nan)
    return out.replace([np.inf, -np.inf], np.nan)


def _add_feature(panel: pd.DataFrame, specs: list[FeatureSpec], name: str, values: pd.Series, *, family: str, variant: str, expected_direction: str, source: str) -> None:
    panel[name] = values.replace([np.inf, -np.inf], np.nan)
    specs.append(FeatureSpec(name, family, variant, expected_direction, source))


In [3]:
MCAP_VALUE_COLUMNS = {
    "revenue": ("income", "revenue"),
    "gross_profit": ("income", "gross_profit"),
    "ebitda": ("income", "ebitda"),
    "ebit": ("income", "ebit"),
    "operating_income": ("income", "operating_income"),
    "net_income": ("income", "net_income"),
    "operating_cash_flow": ("cash", "operating_cash_flow"),
    "free_cash_flow": ("cash", "free_cash_flow"),
    "book_equity": ("balance", "total_stockholders_equity"),
    "tangible_book": ("derived", "tangible_book"),
    "cash": ("balance", "cash_and_cash_equivalents"),
    "cash_and_short_term_investments": ("balance", "cash_and_short_term_investments"),
    "total_debt": ("balance", "total_debt"),
    "net_debt": ("balance", "net_debt"),
}

EV_DENOMINATORS = {
    "revenue": ("income", "revenue"),
    "gross_profit": ("income", "gross_profit"),
    "ebitda": ("income", "ebitda"),
    "ebit": ("income", "ebit"),
    "operating_income": ("income", "operating_income"),
    "operating_cash_flow": ("cash", "operating_cash_flow"),
    "free_cash_flow": ("cash", "free_cash_flow"),
}

STATIC_VENDOR_RATIOS = {
    "static_pe": ("ratios", "price_to_earnings", "lower_is_better"),
    "static_ps": ("ratios", "price_to_sales", "lower_is_better"),
    "static_pb": ("ratios", "price_to_book", "lower_is_better"),
    "static_pocf": ("ratios", "price_to_operating_cash_flow", "lower_is_better"),
    "static_pfcf": ("ratios", "price_to_free_cash_flow", "lower_is_better"),
    "static_debt_to_market_cap": ("ratios", "debt_to_market_cap", "lower_is_better"),
    "static_dividend_yield": ("ratios", "dividend_yield", "higher_is_better"),
    "static_ev_to_sales": ("metrics", "ev_to_sales", "lower_is_better"),
    "static_ev_to_ebitda": ("metrics", "ev_to_ebitda", "lower_is_better"),
    "static_ev_to_ocf": ("metrics", "ev_to_operating_cash_flow", "lower_is_better"),
    "static_ev_to_fcf": ("metrics", "ev_to_free_cash_flow", "lower_is_better"),
}

ASOF_CONTROLS = {
    "gross_profit_margin": ("ratios", "gross_profit_margin", "higher_is_better"),
    "ebitda_margin": ("ratios", "ebitda_margin", "higher_is_better"),
    "operating_profit_margin": ("ratios", "operating_profit_margin", "higher_is_better"),
    "net_profit_margin": ("ratios", "net_profit_margin", "higher_is_better"),
    "current_ratio": ("ratios", "current_ratio", "higher_is_better"),
    "quick_ratio": ("ratios", "quick_ratio", "higher_is_better"),
    "debt_to_assets": ("ratios", "debt_to_assets", "lower_is_better"),
    "debt_to_equity": ("ratios", "debt_to_equity", "lower_is_better"),
    "return_on_assets": ("metrics", "return_on_assets", "higher_is_better"),
    "return_on_equity": ("metrics", "return_on_equity", "higher_is_better"),
    "return_on_invested_capital": ("metrics", "return_on_invested_capital", "higher_is_better"),
    "income_quality": ("metrics", "income_quality", "higher_is_better"),
    "net_debt_to_ebitda": ("metrics", "net_debt_to_ebitda", "lower_is_better"),
}


In [4]:
def load_symbol_inputs(symbol: str) -> dict[str, pd.DataFrame]:
    return {
        "prices": _slice(wh.read_prices(symbol, provider=PROVIDER), START_DATE, END_DATE),
        "historical_market_cap": _slice(wh.read_fundamentals(symbol, section="historical_market_cap", provider=PROVIDER), START_DATE, END_DATE),
        "income": _slice(wh.read_fundamentals(symbol, section="income", provider=PROVIDER), None, END_DATE),
        "balance": _slice(wh.read_fundamentals(symbol, section="balance", provider=PROVIDER), None, END_DATE),
        "cash": _slice(wh.read_fundamentals(symbol, section="cash", provider=PROVIDER), None, END_DATE),
        "ratios": _slice(wh.read_fundamentals(symbol, section="ratios", provider=PROVIDER), None, END_DATE),
        "metrics": _slice(wh.read_fundamentals(symbol, section="metrics", provider=PROVIDER), None, END_DATE),
    }


def _numeric_frame(frame: pd.DataFrame) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)


def _asof_frame_to_daily(frame: pd.DataFrame, daily_index: pd.DatetimeIndex, *, lag_days: int = FILING_LAG_DAYS) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame(index=daily_index)
    sparse = _numeric_frame(frame)
    sparse.index = pd.DatetimeIndex(pd.to_datetime(sparse.index, errors="coerce")).normalize() + pd.Timedelta(days=int(lag_days))
    sparse = sparse.loc[sparse.index.notna()].sort_index()
    sparse = sparse.loc[~sparse.index.duplicated(keep="last")]
    return sparse.reindex(daily_index, method="ffill")


def _add_tangible_book(balance_daily: pd.DataFrame) -> pd.DataFrame:
    if balance_daily.empty or "total_stockholders_equity" not in balance_daily.columns:
        return balance_daily
    out = balance_daily.copy()
    if "goodwill_and_intangible_assets" in out.columns:
        goodwill_intangible = out["goodwill_and_intangible_assets"]
    else:
        goodwill = out["goodwill"] if "goodwill" in out.columns else 0.0
        intangible = out["intangible_assets"] if "intangible_assets" in out.columns else 0.0
        goodwill_intangible = goodwill + intangible
    out["tangible_book"] = out["total_stockholders_equity"] - goodwill_intangible
    return out


def _daily_column(aligned_sources: dict[str, pd.DataFrame], source: str, column: str) -> pd.Series:
    frame = aligned_sources.get(source)
    if frame is None or frame.empty or column not in frame.columns:
        return pd.Series(dtype="float64")
    return frame[column]


def build_symbol_panel(symbol: str) -> tuple[pd.DataFrame, list[FeatureSpec], dict[str, object]]:
    inputs = load_symbol_inputs(symbol)
    prices = inputs["prices"]
    mcap = inputs["historical_market_cap"]
    if prices.empty or mcap.empty or "close" not in prices.columns or "market_cap" not in mcap.columns:
        return pd.DataFrame(), [], {"symbol": symbol, "status": "missing_prices_or_market_cap"}

    close = pd.to_numeric(prices["close"], errors="coerce")
    daily_index = pd.DatetimeIndex(prices.index)
    daily_mcap = pd.to_numeric(mcap["market_cap"], errors="coerce").reindex(daily_index, method="ffill")
    panel = pd.DataFrame({"date": daily_index, "symbol": symbol, "close": close, "daily_market_cap": daily_mcap}, index=daily_index)
    specs: list[FeatureSpec] = []

    align_start = perf_counter()
    aligned_sources = {
        "income": _asof_frame_to_daily(inputs["income"], daily_index),
        "balance": _add_tangible_book(_asof_frame_to_daily(inputs["balance"], daily_index)),
        "cash": _asof_frame_to_daily(inputs["cash"], daily_index),
        "ratios": _asof_frame_to_daily(inputs["ratios"], daily_index),
        "metrics": _asof_frame_to_daily(inputs["metrics"], daily_index),
    }
    align_seconds = perf_counter() - align_start

    cash = _daily_column(aligned_sources, "balance", "cash_and_cash_equivalents")
    total_debt = _daily_column(aligned_sources, "balance", "total_debt")
    net_debt = _daily_column(aligned_sources, "balance", "net_debt")
    panel["daily_enterprise_value"] = daily_mcap + total_debt.reindex(daily_index) - cash.reindex(daily_index)
    panel["asof_total_debt"] = total_debt.reindex(daily_index)
    panel["asof_cash"] = cash.reindex(daily_index)
    panel["asof_net_debt"] = net_debt.reindex(daily_index)

    for label, (source, column) in MCAP_VALUE_COLUMNS.items():
        source_name = "balance" if source == "derived" else source
        daily_value = _daily_column(aligned_sources, source_name, column)
        if daily_value.empty or daily_value.notna().sum() < MIN_OBS:
            continue
        yield_name = f"daily_{label}_to_mcap"
        multiple_name = f"daily_mcap_to_{label}"
        _add_feature(panel, specs, yield_name, _safe_divide(daily_value, daily_mcap), family=label, variant="daily_market_cap_yield", expected_direction="higher_is_better", source=f"{source}.{column}")
        _add_feature(panel, specs, multiple_name, _safe_divide(daily_mcap, daily_value), family=label, variant="daily_market_cap_multiple", expected_direction="lower_is_better", source=f"{source}.{column}")

    for label, (source, column) in EV_DENOMINATORS.items():
        daily_value = _daily_column(aligned_sources, source, column)
        if daily_value.empty or daily_value.notna().sum() < MIN_OBS:
            continue
        multiple_name = f"daily_ev_to_{label}"
        yield_name = f"daily_{label}_to_ev"
        _add_feature(panel, specs, multiple_name, _safe_divide(panel["daily_enterprise_value"], daily_value), family=f"ev_{label}", variant="daily_ev_multiple", expected_direction="lower_is_better", source=f"{source}.{column}")
        _add_feature(panel, specs, yield_name, _safe_divide(daily_value, panel["daily_enterprise_value"]), family=f"ev_{label}", variant="daily_ev_yield", expected_direction="higher_is_better", source=f"{source}.{column}")

    for name, (source, column, direction) in STATIC_VENDOR_RATIOS.items():
        values = _daily_column(aligned_sources, source, column)
        if values.empty or values.notna().sum() < MIN_OBS:
            continue
        _add_feature(panel, specs, name, values, family=name.replace("static_", ""), variant="static_vendor_ratio", expected_direction=direction, source=f"{source}.{column}")

    for name, (source, column, direction) in ASOF_CONTROLS.items():
        values = _daily_column(aligned_sources, source, column)
        if values.empty or values.notna().sum() < MIN_OBS:
            continue
        _add_feature(panel, specs, f"asof_{name}", values, family=name, variant="asof_quality_control", expected_direction=direction, source=f"{source}.{column}")

    for horizon in HORIZONS:
        panel[f"forward_return_{horizon}d"] = close.shift(-int(horizon)) / close - 1.0

    diagnostics = {
        "symbol": symbol,
        "status": "ok",
        "price_rows": len(prices),
        "market_cap_rows": len(mcap),
        "market_cap_start": str(mcap.index.min().date()),
        "market_cap_end": str(mcap.index.max().date()),
        "income_rows": len(inputs["income"]),
        "balance_rows": len(inputs["balance"]),
        "cash_rows": len(inputs["cash"]),
        "feature_count": len(specs),
        "source_align_seconds": align_seconds,
    }
    return panel.reset_index(drop=True), specs, diagnostics


panel_build_start = perf_counter()
frames: list[pd.DataFrame] = []
all_specs: list[FeatureSpec] = []
diagnostics: list[dict[str, object]] = []
for symbol in ANALYSIS_SYMBOLS:
    frame, specs, diag = build_symbol_panel(symbol)
    diagnostics.append(diag)
    if not frame.empty:
        frames.append(frame)
        all_specs.extend(specs)

panel = pd.concat(frames, ignore_index=True).sort_values(["date", "symbol"]).reset_index(drop=True)
feature_metadata = pd.DataFrame([spec.__dict__ for spec in all_specs]).drop_duplicates().sort_values(["variant", "family", "feature"]).reset_index(drop=True)
diagnostics_df = pd.DataFrame(diagnostics)
feature_cols = feature_metadata["feature"].tolist()
run_timings["panel_build_seconds"] = perf_counter() - panel_build_start
run_timings["source_align_seconds"] = float(diagnostics_df.get("source_align_seconds", pd.Series(dtype="float64")).sum())

print({"symbols": sorted(panel["symbol"].unique()), "rows": len(panel), "features": len(feature_cols), "start": str(panel["date"].min().date()), "end": str(panel["date"].max().date()), "panel_build_seconds": round(run_timings["panel_build_seconds"], 2), "source_align_seconds": round(run_timings["source_align_seconds"], 2)})
display(diagnostics_df)
display(feature_metadata.groupby(["variant", "expected_direction"]).size().rename("feature_count").reset_index())
display(feature_metadata.head(30))


{'symbols': ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA'], 'rows': 14910, 'features': 59, 'start': '2018-01-02', 'end': '2026-06-24', 'panel_build_seconds': 0.28, 'source_align_seconds': 0.06}


,symbol,status,price_rows,market_cap_rows,market_cap_start,market_cap_end,income_rows,balance_rows,cash_rows,feature_count,source_align_seconds
0,AAPL,ok,2130,2127,2018-01-02,2026-06-18,162,150,145,59,0.0111
1,MSFT,ok,2130,2127,2018-01-02,2026-06-18,162,151,147,59,0.0076
2,NVDA,ok,2130,2127,2018-01-02,2026-06-18,109,109,109,59,0.0091
3,AMZN,ok,2130,2127,2018-01-02,2026-06-18,121,118,117,59,0.0073
4,META,ok,2130,2127,2018-01-02,2026-06-18,61,58,61,59,0.0069
5,GOOGL,ok,2130,2127,2018-01-02,2026-06-18,97,90,93,59,0.0072
6,TSLA,ok,2130,2127,2018-01-02,2026-06-18,73,72,73,59,0.0070


,variant,expected_direction,feature_count
0,asof_quality_control,higher_is_better,10
1,asof_quality_control,lower_is_better,1
2,daily_ev_multiple,lower_is_better,7
3,daily_ev_yield,higher_is_better,7
4,daily_market_cap_multiple,lower_is_better,14
5,daily_market_cap_yield,higher_is_better,14
6,static_vendor_ratio,higher_is_better,1
7,static_vendor_ratio,lower_is_better,5


,feature,family,variant,expected_direction,source
0,asof_current_ratio,current_ratio,asof_quality_control,higher_is_better,ratios.current_ratio
1,asof_ebitda_margin,ebitda_margin,asof_quality_control,higher_is_better,ratios.ebitda_margin
2,asof_gross_profit_margin,gross_profit_margin,asof_quality_control,higher_is_better,ratios.gross_profit_margin
3,asof_income_quality,income_quality,asof_quality_control,higher_is_better,metrics.income_quality
4,asof_net_debt_to_ebitda,net_debt_to_ebitda,asof_quality_control,lower_is_better,metrics.net_debt_to_ebitda
5,asof_net_profit_margin,net_profit_margin,asof_quality_control,higher_is_better,ratios.net_profit_margin
6,asof_operating_profit_margin,operating_profit_margin,asof_quality_control,higher_is_better,ratios.operating_profit_margin
7,asof_quick_ratio,quick_ratio,asof_quality_control,higher_is_better,ratios.quick_ratio
8,asof_return_on_assets,return_on_assets,asof_quality_control,higher_is_better,metrics.return_on_assets
9,asof_return_on_equity,return_on_equity,asof_quality_control,higher_is_better,metrics.return_on_equity


## Coverage And Sanity Checks

Before measuring signal, verify that the daily market cap source is actually daily and that the reconstructed EV is available after the first lagged balance sheet observation.


In [5]:
coverage_rows = []
for symbol, group in panel.groupby("symbol"):
    group = group.sort_values("date")
    mcap = group.dropna(subset=["daily_market_cap"])
    ev = group.dropna(subset=["daily_enterprise_value"])
    gaps = pd.to_datetime(mcap["date"]).diff().dt.days.dropna()
    coverage_rows.append({
        "symbol": symbol,
        "panel_rows": len(group),
        "mcap_non_null": int(group["daily_market_cap"].notna().sum()),
        "ev_non_null": int(group["daily_enterprise_value"].notna().sum()),
        "mcap_start": mcap["date"].min(),
        "mcap_end": mcap["date"].max(),
        "mcap_median_gap_days": float(gaps.median()) if not gaps.empty else np.nan,
        "mcap_mode_gap_days": int(gaps.mode().iloc[0]) if not gaps.empty else np.nan,
        "latest_market_cap": mcap["daily_market_cap"].iloc[-1] if not mcap.empty else np.nan,
        "latest_daily_ev": ev["daily_enterprise_value"].iloc[-1] if not ev.empty else np.nan,
    })
coverage = pd.DataFrame(coverage_rows).sort_values("symbol")
display(coverage)

sample_cols = ["symbol", "date", "close", "daily_market_cap", "daily_enterprise_value", "asof_total_debt", "asof_cash"]
display(panel[sample_cols].dropna().groupby("symbol").tail(2))


,symbol,panel_rows,mcap_non_null,ev_non_null,mcap_start,mcap_end,mcap_median_gap_days,mcap_mode_gap_days,latest_market_cap,latest_daily_ev
0,AAPL,2130,2130,2130,2018-01-02,2026-06-24,1.0000,1,4383941071180,4432324071180
1,AMZN,2130,2130,2130,2018-01-02,2026-06-24,1.0000,1,2625481770000,2733553770000
2,GOOGL,2130,2130,2130,2018-01-02,2026-06-24,1.0000,1,4452794970000,4505215970000
3,META,2130,2130,2130,2018-01-02,2026-06-24,1.0000,1,1462675480000,1526018480000
4,MSFT,2130,2130,2130,2018-01-02,2026-06-24,1.0000,1,2817424400000,2842284400000
5,NVDA,2130,2130,2130,2018-01-02,2026-06-24,1.0000,1,5116817340000,5117624340000
6,TSLA,2130,2130,2130,2018-01-02,2026-06-24,1.0000,1,1295184660000,1287810660000


,symbol,date,close,daily_market_cap,daily_enterprise_value,asof_total_debt,asof_cash
14896,AAPL,2026-06-23,294.3000,4383941071180,4432324071180,84711000000,36328000000
14897,AMZN,2026-06-23,234.1100,2625481770000,2733553770000,209888000000,101816000000
14898,GOOGL,2026-06-23,346.1300,4452794970000,4505215970000,90484000000,38063000000
14899,META,2026-06-23,562.2000,1462675480000,1526018480000,86769000000,23426000000
14900,MSFT,2026-06-23,373.9400,2817424400000,2842284400000,56965000000,32105000000
14901,NVDA,2026-06-23,200.0400,5116817340000,5117624340000,11412000000,10605000000
14902,TSLA,2026-06-23,381.6100,1295184660000,1287810660000,9229000000,16603000000
14903,AAPL,2026-06-24,293.0800,4383941071180,4432324071180,84711000000,36328000000
14904,AMZN,2026-06-24,234.2700,2625481770000,2733553770000,209888000000,101816000000
14905,GOOGL,2026-06-24,345.2900,4452794970000,4505215970000,90484000000,38063000000


## Correctness And Runtime Metrics

For the smoke run, the actual small-sample rank IC values are not the point. This section checks that:

- The full feature panel can be evaluated end to end.
- A simple baseline evaluator and the optimized evaluator produce matching rank IC values on a subset.
- The optimized evaluator is fast enough to justify scaling to more symbols or longer history.

The baseline subset intentionally uses only a few features to keep the notebook quick. The optimized evaluator runs all engineered features.

In [6]:
def _spearman(a: pd.Series, b: pd.Series) -> float:
    valid = pd.concat([a, b], axis=1).dropna()
    if len(valid) < 3 or valid.iloc[:, 0].nunique() < 2 or valid.iloc[:, 1].nunique() < 2:
        return np.nan
    return float(valid.iloc[:, 0].rank().corr(valid.iloc[:, 1].rank()))


def _score_direction(values: pd.Series | pd.DataFrame, expected_direction: str):
    if expected_direction == "lower_is_better":
        return -values
    return values


def _rank_2d_nan(values: np.ndarray) -> np.ndarray:
    """Rank each row, preserving NaNs. Small row width, optimized for correctness/readability."""
    out = np.full(values.shape, np.nan, dtype="float32")
    for i in range(values.shape[0]):
        row = values[i]
        valid = np.isfinite(row)
        count = int(valid.sum())
        if count == 0:
            continue
        order = np.argsort(row[valid], kind="mergesort")
        ranks = np.empty(count, dtype="float32")
        ranks[order] = np.arange(1, count + 1, dtype="float32")
        out[i, np.flatnonzero(valid)] = ranks
    return out


def _rank_3d_by_symbol(values: np.ndarray) -> np.ndarray:
    days, symbols, features = values.shape
    flat = values.transpose(0, 2, 1).reshape(days * features, symbols)
    ranked = _rank_2d_nan(flat)
    return ranked.reshape(days, features, symbols).transpose(0, 2, 1)


def _mean_center_nan(values: np.ndarray, axis: int) -> np.ndarray:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean = np.nanmean(values, axis=axis, keepdims=True)
    return values - mean


def _build_eval_arrays(features: list[str]) -> tuple[np.ndarray, dict[int, np.ndarray], pd.DatetimeIndex, list[str]]:
    dates = pd.DatetimeIndex(sorted(panel["date"].dropna().unique()))
    symbols = sorted(panel["symbol"].dropna().unique())
    date_index = {date: i for i, date in enumerate(dates)}
    symbol_index = {symbol: i for i, symbol in enumerate(symbols)}

    feature_values = np.full((len(dates), len(symbols), len(features)), np.nan, dtype="float32")
    for feature_idx, feature in enumerate(features):
        pivot = panel.pivot(index="date", columns="symbol", values=feature).reindex(index=dates, columns=symbols)
        feature_values[:, :, feature_idx] = pivot.to_numpy(dtype="float32")

    direction_sign = feature_metadata.set_index("feature").loc[features, "expected_direction"].map({"higher_is_better": 1.0, "lower_is_better": -1.0}).to_numpy(dtype="float32")
    feature_scores = feature_values * direction_sign.reshape(1, 1, -1)

    returns_by_horizon: dict[int, np.ndarray] = {}
    for horizon in HORIZONS:
        pivot = panel.pivot(index="date", columns="symbol", values=f"forward_return_{horizon}d").reindex(index=dates, columns=symbols)
        returns_by_horizon[horizon] = pivot.to_numpy(dtype="float32")
    return feature_scores, returns_by_horizon, dates, symbols


def evaluate_array(features: list[str], horizons: tuple[int, ...]) -> pd.DataFrame:
    build_start = perf_counter()
    feature_scores, returns_by_horizon, dates, symbols = _build_eval_arrays(features)
    run_timings["eval_array_build_seconds"] = perf_counter() - build_start

    rank_start = perf_counter()
    feature_ranks = _rank_3d_by_symbol(feature_scores)
    run_timings["feature_rank_seconds"] = perf_counter() - rank_start

    rows = []
    for horizon in horizons:
        horizon_start = perf_counter()
        returns = returns_by_horizon[horizon]
        return_ranks = _rank_2d_nan(returns)

        centered_features = _mean_center_nan(feature_ranks, axis=1)
        centered_returns = _mean_center_nan(return_ranks, axis=1)
        valid = np.isfinite(centered_features) & np.isfinite(centered_returns[:, :, None])
        cf = np.where(valid, centered_features, 0.0)
        cr = np.where(np.isfinite(centered_returns), centered_returns, 0.0)
        numerator = np.einsum("dsf,ds->df", cf, cr, optimize=True)
        feature_ss = np.einsum("dsf,dsf->df", cf, cf, optimize=True)
        return_ss = np.einsum("ds,ds->d", cr, cr, optimize=True)
        denominator = np.sqrt(feature_ss * return_ss[:, None])
        with np.errstate(invalid="ignore", divide="ignore"):
            daily_ic = numerator / denominator
        daily_ic[denominator == 0] = np.nan

        score_ranks = feature_ranks
        valid_counts = np.isfinite(score_ranks).sum(axis=1).astype("float32")
        n_by_feature = np.where(valid_counts < 4, 1.0, 2.0)
        top_mask = score_ranks > (valid_counts[:, None, :] - n_by_feature[:, None, :])
        bottom_mask = score_ranks <= n_by_feature[:, None, :]
        returns_3d = returns[:, :, None]
        top_returns = np.where(top_mask & np.isfinite(returns_3d), returns_3d, np.nan)
        bottom_returns = np.where(bottom_mask & np.isfinite(returns_3d), returns_3d, np.nan)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            daily_spread = np.nanmean(top_returns, axis=1) - np.nanmean(bottom_returns, axis=1)
        daily_spread[valid_counts < 3] = np.nan

        for feature_idx, feature in enumerate(features):
            feature_values = feature_scores[:, :, feature_idx]
            return_values = returns
            valid_obs = np.isfinite(feature_values) & np.isfinite(return_values)
            pooled_score = feature_values[valid_obs]
            pooled_return = return_values[valid_obs]
            ic_series = pd.Series(daily_ic[:, feature_idx], dtype="float64").dropna()
            spread_series = pd.Series(daily_spread[:, feature_idx], dtype="float64").dropna()
            rows.append({
                "feature": feature,
                "horizon": horizon,
                "observations": int(valid_obs.sum()),
                "days": int(ic_series.size),
                "pooled_spearman": _spearman(pd.Series(pooled_score), pd.Series(pooled_return)),
                "mean_daily_rank_ic": float(ic_series.mean()) if not ic_series.empty else np.nan,
                "median_daily_rank_ic": float(ic_series.median()) if not ic_series.empty else np.nan,
                "rank_ic_hit_rate": float((ic_series > 0).mean()) if not ic_series.empty else np.nan,
                "mean_top_minus_bottom": float(spread_series.mean()) if not spread_series.empty else np.nan,
                "median_top_minus_bottom": float(spread_series.median()) if not spread_series.empty else np.nan,
            })
        run_timings[f"eval_horizon_{horizon}_seconds"] = perf_counter() - horizon_start
    return pd.DataFrame(rows)


def evaluate_baseline_loop(features: list[str], horizons: tuple[int, ...]) -> pd.DataFrame:
    rows = []
    direction_sign = feature_metadata.set_index("feature").loc[features, "expected_direction"].map({"higher_is_better": 1.0, "lower_is_better": -1.0})
    score_frame = panel[["date", "symbol", *features]].copy()
    score_frame.loc[:, features] = score_frame[features].multiply(direction_sign, axis="columns")
    for horizon in horizons:
        return_col = f"forward_return_{horizon}d"
        eval_frame = score_frame[["date", "symbol", *features]].copy()
        eval_frame[return_col] = panel[return_col]
        for feature in features:
            daily_ic = []
            daily_spread = []
            for _, group in eval_frame[["date", "symbol", feature, return_col]].groupby("date", sort=False):
                valid = group[[feature, return_col]].replace([np.inf, -np.inf], np.nan).dropna()
                if len(valid) < 3 or valid[feature].nunique() < 2 or valid[return_col].nunique() < 2:
                    continue
                daily_ic.append(_spearman(valid[feature], valid[return_col]))
                n = 1 if len(valid) < 4 else 2
                ranked = valid.sort_values(feature)
                daily_spread.append(ranked.tail(n)[return_col].mean() - ranked.head(n)[return_col].mean())
            daily_ic_s = pd.Series(daily_ic, dtype="float64").dropna()
            daily_spread_s = pd.Series(daily_spread, dtype="float64").dropna()
            valid_all = eval_frame[[feature, return_col]].replace([np.inf, -np.inf], np.nan).dropna()
            rows.append({
                "feature": feature,
                "horizon": horizon,
                "observations": int(len(valid_all)),
                "days": int(len(daily_ic_s)),
                "pooled_spearman": _spearman(valid_all[feature], valid_all[return_col]),
                "mean_daily_rank_ic": float(daily_ic_s.mean()) if not daily_ic_s.empty else np.nan,
                "median_daily_rank_ic": float(daily_ic_s.median()) if not daily_ic_s.empty else np.nan,
                "rank_ic_hit_rate": float((daily_ic_s > 0).mean()) if not daily_ic_s.empty else np.nan,
                "mean_top_minus_bottom": float(daily_spread_s.mean()) if not daily_spread_s.empty else np.nan,
                "median_top_minus_bottom": float(daily_spread_s.median()) if not daily_spread_s.empty else np.nan,
            })
    return pd.DataFrame(rows)


benchmark_features = feature_cols[: min(6, len(feature_cols))]
benchmark_horizons = (HORIZONS[0],)
baseline_start = perf_counter()
baseline_subset_results = evaluate_baseline_loop(benchmark_features, benchmark_horizons)
run_timings["baseline_subset_seconds"] = perf_counter() - baseline_start

optimized_subset_start = perf_counter()
optimized_subset_results = evaluate_array(benchmark_features, benchmark_horizons)
run_timings["optimized_subset_seconds"] = perf_counter() - optimized_subset_start

optimized_full_start = perf_counter()
results_df = evaluate_array(feature_cols, HORIZONS)
run_timings["evaluation_seconds"] = perf_counter() - optimized_full_start

results_df = results_df.merge(feature_metadata, on="feature", how="left")
results_df["abs_mean_daily_rank_ic"] = results_df["mean_daily_rank_ic"].abs()
results_df["spread_bps"] = results_df["mean_top_minus_bottom"] * 10000
results_df = results_df.dropna(subset=["mean_daily_rank_ic"]).reset_index(drop=True)

benchmark_compare = baseline_subset_results.merge(
    optimized_subset_results,
    on=["feature", "horizon"],
    suffixes=("_baseline", "_optimized"),
)
max_metric_diff = (
    benchmark_compare["mean_daily_rank_ic_baseline"] - benchmark_compare["mean_daily_rank_ic_optimized"]
).abs().max()
max_spread_diff = (
    benchmark_compare["mean_top_minus_bottom_baseline"] - benchmark_compare["mean_top_minus_bottom_optimized"]
).abs().max()

runtime_benchmark = pd.DataFrame([
    {
        "method": "baseline_loop_correctness_subset",
        "features": len(benchmark_features),
        "horizons": len(benchmark_horizons),
        "seconds": run_timings["baseline_subset_seconds"],
        "seconds_per_feature_horizon": run_timings["baseline_subset_seconds"] / max(len(benchmark_features) * len(benchmark_horizons), 1),
    },
    {
        "method": "array_evaluator_correctness_subset",
        "features": len(benchmark_features),
        "horizons": len(benchmark_horizons),
        "seconds": run_timings["optimized_subset_seconds"],
        "seconds_per_feature_horizon": run_timings["optimized_subset_seconds"] / max(len(benchmark_features) * len(benchmark_horizons), 1),
    },
    {
        "method": "array_evaluator_full",
        "features": len(feature_cols),
        "horizons": len(HORIZONS),
        "seconds": run_timings["evaluation_seconds"],
        "seconds_per_feature_horizon": run_timings["evaluation_seconds"] / max(len(feature_cols) * len(HORIZONS), 1),
    },
])

summary_by_variant = (
    results_df.groupby(["horizon", "variant"])
    .agg(
        features=("feature", "nunique"),
        median_rank_ic=("mean_daily_rank_ic", "median"),
        mean_rank_ic=("mean_daily_rank_ic", "mean"),
        median_spread_bps=("spread_bps", "median"),
        positive_ic_share=("mean_daily_rank_ic", lambda x: float((x > 0).mean())),
    )
    .reset_index()
    .sort_values(["horizon", "median_rank_ic"], ascending=[True, False])
)

top_features = (
    results_df.sort_values(["horizon", "mean_daily_rank_ic", "mean_top_minus_bottom"], ascending=[True, False, False])
    .groupby("horizon")
    .head(15)
    [["horizon", "feature", "variant", "family", "expected_direction", "mean_daily_rank_ic", "median_daily_rank_ic", "rank_ic_hit_rate", "spread_bps", "observations"]]
)

worst_features = (
    results_df.sort_values(["horizon", "mean_daily_rank_ic", "mean_top_minus_bottom"], ascending=[True, True, True])
    .groupby("horizon")
    .head(10)
    [["horizon", "feature", "variant", "family", "expected_direction", "mean_daily_rank_ic", "median_daily_rank_ic", "rank_ic_hit_rate", "spread_bps", "observations"]]
)

print({
    "array_evaluator_full_seconds": round(run_timings["evaluation_seconds"], 3),
    "baseline_correctness_subset_seconds": round(run_timings["baseline_subset_seconds"], 3),
    "array_correctness_subset_seconds": round(run_timings["optimized_subset_seconds"], 3),
    "max_rank_ic_diff_baseline_vs_array": None if pd.isna(max_metric_diff) else round(float(max_metric_diff), 12),
    "max_spread_diff_baseline_vs_array": None if pd.isna(max_spread_diff) else round(float(max_spread_diff), 12),
    "feature_horizon_results": len(results_df),
})
display(runtime_benchmark)
display(summary_by_variant)
display(top_features)
display(worst_features)


{'array_evaluator_full_seconds': 0.914, 'baseline_correctness_subset_seconds': 7.722, 'array_correctness_subset_seconds': 0.075, 'max_rank_ic_diff_baseline_vs_array': 5.4e-10, 'max_spread_diff_baseline_vs_array': 3.01e-10, 'feature_horizon_results': 177}


,method,features,horizons,seconds,seconds_per_feature_horizon
0,baseline_loop_correctness_subset,6,1,7.7217,1.2869
1,array_evaluator_correctness_subset,6,1,0.0750,0.0125
2,array_evaluator_full,59,3,0.9140,0.0052


,horizon,variant,features,median_rank_ic,mean_rank_ic,median_spread_bps,positive_ic_share
3,20,daily_market_cap_multiple,14,-0.0001,-0.0063,-120.5175,0.5000
0,20,asof_quality_control,11,-0.0018,-0.0006,-112.7251,0.3636
5,20,static_vendor_ratio,6,-0.0028,-0.0015,-70.3899,0.5000
4,20,daily_market_cap_yield,14,-0.0032,-0.0026,-132.1976,0.4286
1,20,daily_ev_multiple,7,-0.0093,-0.0096,-138.6825,0.4286
2,20,daily_ev_yield,7,-0.0100,-0.0126,-199.1534,0.1429
6,60,asof_quality_control,11,0.0013,0.0077,-300.6758,0.5455
11,60,static_vendor_ratio,6,-0.0101,-0.0078,-158.4027,0.3333
9,60,daily_market_cap_multiple,14,-0.0212,-0.0246,-295.7434,0.3571
10,60,daily_market_cap_yield,14,-0.0253,-0.0193,-364.4989,0.2857


,horizon,feature,variant,family,expected_direction,mean_daily_rank_ic,median_daily_rank_ic,rank_ic_hit_rate,spread_bps,observations
7,20,asof_quick_ratio,asof_quality_control,quick_ratio,higher_is_better,0.0448,0.0357,0.5194,19.7899,14770
54,20,static_dividend_yield,static_vendor_ratio,dividend_yield,higher_is_better,0.0403,0.0357,0.5232,13.6281,14770
0,20,asof_current_ratio,asof_quality_control,current_ratio,higher_is_better,0.0374,0.0357,0.5156,49.1229,14770
52,20,daily_total_debt_to_mcap,daily_market_cap_yield,total_debt,higher_is_better,0.0251,0.0000,0.4905,107.9262,14770
38,20,daily_mcap_to_total_debt,daily_market_cap_multiple,total_debt,lower_is_better,0.0195,0.0000,0.4853,100.6778,14718
57,20,static_ev_to_ocf,static_vendor_ratio,ev_to_ocf,lower_is_better,0.0183,0.0357,0.5047,-78.7676,14770
34,20,daily_mcap_to_operating_cash_flow,daily_market_cap_multiple,operating_cash_flow,lower_is_better,0.0167,0.0000,0.4896,-64.2140,14770
26,20,daily_mcap_to_cash,daily_market_cap_multiple,cash,lower_is_better,0.0162,0.0000,0.4791,101.5312,14770
40,20,daily_cash_to_mcap,daily_market_cap_yield,cash,higher_is_better,0.0162,0.0000,0.4791,101.5312,14770
15,20,daily_ev_to_operating_cash_flow,daily_ev_multiple,ev_operating_cash_flow,lower_is_better,0.0118,0.0000,0.4829,-82.9814,14770


,horizon,feature,variant,family,expected_direction,mean_daily_rank_ic,median_daily_rank_ic,rank_ic_hit_rate,spread_bps,observations
32,20,daily_mcap_to_net_debt,daily_market_cap_multiple,net_debt,lower_is_better,-0.0684,-0.0714,0.4014,-151.9997,14770
53,20,static_debt_to_market_cap,static_vendor_ratio,debt_to_market_cap,lower_is_better,-0.0414,-0.0357,0.4332,-149.7031,14770
31,20,daily_mcap_to_gross_profit,daily_market_cap_multiple,gross_profit,lower_is_better,-0.0402,-0.0357,0.4607,-259.2414,14770
45,20,daily_gross_profit_to_mcap,daily_market_cap_yield,gross_profit,higher_is_better,-0.0402,-0.0357,0.4607,-259.2414,14770
14,20,daily_ev_to_gross_profit,daily_ev_multiple,ev_gross_profit,lower_is_better,-0.0392,-0.0357,0.4645,-258.7741,14770
21,20,daily_gross_profit_to_ev,daily_ev_yield,ev_gross_profit,higher_is_better,-0.0392,-0.0357,0.4645,-258.7741,14770
3,20,asof_income_quality,asof_quality_control,income_quality,higher_is_better,-0.0373,-0.0357,0.4550,-25.0629,14770
6,20,asof_operating_profit_margin,asof_quality_control,operating_profit_margin,higher_is_better,-0.0255,-0.0357,0.4507,-202.6108,14770
13,20,daily_ev_to_free_cash_flow,daily_ev_multiple,ev_free_cash_flow,lower_is_better,-0.0203,0.0000,0.4697,-96.2731,14770
55,20,static_ev_to_ebitda,static_vendor_ratio,ev_to_ebitda,lower_is_better,-0.0203,0.0000,0.4749,-183.3240,14770


## Validation Method Tradeoffs

The full evaluator is not the only way to test the daily-adjusted-fundamental hypothesis. For larger universes, a cheaper validation pass may be preferable if it lets us screen hundreds of symbols quickly.

This section compares three validation tiers:

- **Fast pooled Pearson**: cheapest directional screen across all symbol-days; less robust because it ignores daily cross-sectional structure.
- **Rank IC only**: daily cross-sectional rank correlation; good middle ground and usually the best first-pass research metric.
- **Full array evaluator**: rank IC plus top-minus-bottom spread and pooled diagnostics; most informative but not always necessary for early screening.


In [7]:
def validate_fast_pooled_pearson(features: list[str], horizons: tuple[int, ...]) -> pd.DataFrame:
    feature_scores, returns_by_horizon, _, _ = _build_eval_arrays(features)
    x = feature_scores.reshape(-1, len(features)).astype("float64")
    rows = []
    for horizon in horizons:
        y = returns_by_horizon[horizon].reshape(-1).astype("float64")
        valid = np.isfinite(x) & np.isfinite(y[:, None])
        count = valid.sum(axis=0)
        x_sum = np.where(valid, x, 0.0).sum(axis=0)
        y_sum = np.where(valid, y[:, None], 0.0).sum(axis=0)
        x_mean = x_sum / np.maximum(count, 1)
        y_mean = y_sum / np.maximum(count, 1)
        x_centered = np.where(valid, x - x_mean, 0.0)
        y_centered = np.where(valid, y[:, None] - y_mean, 0.0)
        numerator = (x_centered * y_centered).sum(axis=0)
        denominator = np.sqrt((x_centered * x_centered).sum(axis=0) * (y_centered * y_centered).sum(axis=0))
        corr = numerator / denominator
        corr[denominator == 0] = np.nan
        for feature, value, obs in zip(features, corr, count, strict=True):
            rows.append({"method": "fast_pooled_pearson", "feature": feature, "horizon": horizon, "score": float(value), "observations": int(obs)})
    return pd.DataFrame(rows)


def validate_rank_ic_only(features: list[str], horizons: tuple[int, ...]) -> pd.DataFrame:
    feature_scores, returns_by_horizon, _, _ = _build_eval_arrays(features)
    feature_ranks = _rank_3d_by_symbol(feature_scores)
    centered_features = _mean_center_nan(feature_ranks, axis=1)
    rows = []
    for horizon in horizons:
        returns = returns_by_horizon[horizon]
        return_ranks = _rank_2d_nan(returns)
        centered_returns = _mean_center_nan(return_ranks, axis=1)
        valid = np.isfinite(centered_features) & np.isfinite(centered_returns[:, :, None])
        cf = np.where(valid, centered_features, 0.0)
        cr = np.where(np.isfinite(centered_returns), centered_returns, 0.0)
        numerator = np.einsum("dsf,ds->df", cf, cr, optimize=True)
        feature_ss = np.einsum("dsf,dsf->df", cf, cf, optimize=True)
        return_ss = np.einsum("ds,ds->d", cr, cr, optimize=True)
        denominator = np.sqrt(feature_ss * return_ss[:, None])
        with np.errstate(invalid="ignore", divide="ignore"):
            daily_ic = numerator / denominator
        daily_ic[denominator == 0] = np.nan
        mean_ic = np.nanmean(daily_ic, axis=0)
        days = np.isfinite(daily_ic).sum(axis=0)
        for feature, value, day_count in zip(features, mean_ic, days, strict=True):
            rows.append({"method": "rank_ic_only", "feature": feature, "horizon": horizon, "score": float(value), "days": int(day_count)})
    return pd.DataFrame(rows)

validation_timing_rows = []

start = perf_counter()
fast_validation = validate_fast_pooled_pearson(feature_cols, HORIZONS)
validation_timing_rows.append({"method": "fast_pooled_pearson", "seconds": perf_counter() - start, "output": "pooled directional correlation", "tradeoff": "fastest, ignores daily cross-section"})

start = perf_counter()
rank_only_validation = validate_rank_ic_only(feature_cols, HORIZONS)
validation_timing_rows.append({"method": "rank_ic_only", "seconds": perf_counter() - start, "output": "mean daily rank IC", "tradeoff": "middle ground, no spread diagnostics"})

validation_timing_rows.append({"method": "full_array_evaluator", "seconds": run_timings["evaluation_seconds"], "output": "rank IC + spread + pooled diagnostics", "tradeoff": "most complete notebook diagnostic"})
validation_timing = pd.DataFrame(validation_timing_rows)
validation_timing["relative_to_fast"] = validation_timing["seconds"] / validation_timing["seconds"].min()

fast_variant = (
    fast_validation.merge(feature_metadata, on="feature", how="left")
    .groupby(["horizon", "variant"])
    .agg(features=("feature", "nunique"), median_score=("score", "median"), mean_score=("score", "mean"), positive_share=("score", lambda x: float((x > 0).mean())))
    .reset_index()
    .sort_values(["horizon", "median_score"], ascending=[True, False])
)

rank_only_variant = (
    rank_only_validation.merge(feature_metadata, on="feature", how="left")
    .groupby(["horizon", "variant"])
    .agg(features=("feature", "nunique"), median_score=("score", "median"), mean_score=("score", "mean"), positive_share=("score", lambda x: float((x > 0).mean())))
    .reset_index()
    .sort_values(["horizon", "median_score"], ascending=[True, False])
)

method_agreement = (
    rank_only_validation.rename(columns={"score": "rank_ic_only_score"})
    .merge(results_df[["feature", "horizon", "mean_daily_rank_ic"]], on=["feature", "horizon"], how="inner")
)
method_agreement_error = float((method_agreement["rank_ic_only_score"] - method_agreement["mean_daily_rank_ic"]).abs().max())

print({"method_agreement_max_abs_error_rank_only_vs_full": method_agreement_error})
display(validation_timing)
display(fast_variant)
display(rank_only_variant)


{'method_agreement_max_abs_error_rank_only_vs_full': 3.582487503664411e-07}


,method,seconds,output,tradeoff,relative_to_fast
0,fast_pooled_pearson,0.0814,pooled directional correlation,"fastest, ignores daily cross-section",1.0000
1,rank_ic_only,0.5259,mean daily rank IC,"middle ground, no spread diagnostics",6.4577
2,full_array_evaluator,0.9140,rank IC + spread + pooled diagnostics,most complete notebook diagnostic,11.2239


,horizon,variant,features,median_score,mean_score,positive_share
4,20,daily_market_cap_yield,14,0.0134,0.0249,0.7857
3,20,daily_market_cap_multiple,14,0.0032,-0.0085,0.5714
2,20,daily_ev_yield,7,0.0016,0.0063,0.5714
5,20,static_vendor_ratio,6,-0.0076,-0.0154,0.3333
1,20,daily_ev_multiple,7,-0.0186,-0.0157,0.2857
0,20,asof_quality_control,11,-0.0273,-0.0280,0.0909
10,60,daily_market_cap_yield,14,0.0308,0.0389,0.7143
9,60,daily_market_cap_multiple,14,0.0001,-0.0198,0.5000
8,60,daily_ev_yield,7,-0.0004,0.0063,0.4286
11,60,static_vendor_ratio,6,-0.0174,-0.0231,0.3333


,horizon,variant,features,median_score,mean_score,positive_share
3,20,daily_market_cap_multiple,14,-0.0001,-0.0063,0.5000
0,20,asof_quality_control,11,-0.0018,-0.0006,0.3636
5,20,static_vendor_ratio,6,-0.0028,-0.0015,0.5000
4,20,daily_market_cap_yield,14,-0.0032,-0.0026,0.4286
1,20,daily_ev_multiple,7,-0.0093,-0.0096,0.4286
2,20,daily_ev_yield,7,-0.0100,-0.0126,0.1429
6,60,asof_quality_control,11,0.0013,0.0077,0.5455
11,60,static_vendor_ratio,6,-0.0101,-0.0078,0.3333
9,60,daily_market_cap_multiple,14,-0.0212,-0.0246,0.3571
10,60,daily_market_cap_yield,14,-0.0253,-0.0193,0.2857


## Static Vendor Ratios Vs Daily-Adjusted Rebuilds

This table compares similar valuation families where FMP provides a static/as-of ratio and the notebook reconstructs a daily-adjusted equivalent from daily market cap or daily EV.


In [8]:
comparison_map = {
    "pe": ["static_pe", "daily_net_income_to_mcap", "daily_mcap_to_net_income"],
    "ps": ["static_ps", "daily_revenue_to_mcap", "daily_mcap_to_revenue"],
    "pb": ["static_pb", "daily_book_equity_to_mcap", "daily_mcap_to_book_equity"],
    "pocf": ["static_pocf", "daily_operating_cash_flow_to_mcap", "daily_mcap_to_operating_cash_flow"],
    "pfcf": ["static_pfcf", "daily_free_cash_flow_to_mcap", "daily_mcap_to_free_cash_flow"],
    "ev_sales": ["static_ev_to_sales", "daily_ev_to_revenue", "daily_revenue_to_ev"],
    "ev_ebitda": ["static_ev_to_ebitda", "daily_ev_to_ebitda", "daily_ebitda_to_ev"],
    "ev_ocf": ["static_ev_to_ocf", "daily_ev_to_operating_cash_flow", "daily_operating_cash_flow_to_ev"],
    "ev_fcf": ["static_ev_to_fcf", "daily_ev_to_free_cash_flow", "daily_free_cash_flow_to_ev"],
}

comparison_rows = []
for family, features in comparison_map.items():
    for feature in features:
        subset = results_df.loc[results_df["feature"].eq(feature)]
        if subset.empty:
            continue
        for _, row in subset.iterrows():
            comparison_rows.append({
                "comparison_family": family,
                "horizon": row["horizon"],
                "feature": feature,
                "variant": row["variant"],
                "mean_daily_rank_ic": row["mean_daily_rank_ic"],
                "rank_ic_hit_rate": row["rank_ic_hit_rate"],
                "spread_bps": row["spread_bps"],
            })
comparison_df = pd.DataFrame(comparison_rows).sort_values(["comparison_family", "horizon", "mean_daily_rank_ic"], ascending=[True, True, False])
display(comparison_df)

winner_df = (
    comparison_df.sort_values(["comparison_family", "horizon", "mean_daily_rank_ic"], ascending=[True, True, False])
    .groupby(["comparison_family", "horizon"])
    .head(1)
    .reset_index(drop=True)
)
display(winner_df)


,comparison_family,horizon,feature,variant,mean_daily_rank_ic,rank_ic_hit_rate,spread_bps
45,ev_ebitda,20,daily_ebitda_to_ev,daily_ev_yield,-0.0078,0.4787,-218.5283
42,ev_ebitda,20,daily_ev_to_ebitda,daily_ev_multiple,-0.0166,0.4659,-212.9959
39,ev_ebitda,20,static_ev_to_ebitda,static_vendor_ratio,-0.0203,0.4749,-183.3240
46,ev_ebitda,60,daily_ebitda_to_ev,daily_ev_yield,-0.0372,0.4686,-702.2745
40,ev_ebitda,60,static_ev_to_ebitda,static_vendor_ratio,-0.0415,0.4386,-641.2331
...,...,...,...,...,...,...,...
9,ps,20,daily_mcap_to_revenue,daily_market_cap_multiple,0.0057,0.4905,-26.6185
7,ps,60,daily_revenue_to_mcap,daily_market_cap_yield,0.0024,0.4841,-47.7428
10,ps,60,daily_mcap_to_revenue,daily_market_cap_multiple,0.0024,0.4841,-47.7428
8,ps,120,daily_revenue_to_mcap,daily_market_cap_yield,-0.0063,0.4776,46.9671


,comparison_family,horizon,feature,variant,mean_daily_rank_ic,rank_ic_hit_rate,spread_bps
0,ev_ebitda,20,daily_ebitda_to_ev,daily_ev_yield,-0.0078,0.4787,-218.5283
1,ev_ebitda,60,daily_ebitda_to_ev,daily_ev_yield,-0.0372,0.4686,-702.2745
2,ev_ebitda,120,static_ev_to_ebitda,static_vendor_ratio,-0.0346,0.4687,"-1,264.2378"
3,ev_fcf,20,static_ev_to_fcf,static_vendor_ratio,0.0010,0.4995,-62.0123
4,ev_fcf,60,static_ev_to_fcf,static_vendor_ratio,-0.0003,0.5184,-130.6613
5,ev_fcf,120,static_ev_to_fcf,static_vendor_ratio,-0.0103,0.4502,293.9442
6,ev_ocf,20,static_ev_to_ocf,static_vendor_ratio,0.0183,0.5047,-78.7676
7,ev_ocf,60,static_ev_to_ocf,static_vendor_ratio,0.0172,0.4928,-186.1440
8,ev_ocf,120,static_ev_to_ocf,static_vendor_ratio,-0.0077,0.4975,-143.8535
9,ev_sales,20,daily_ev_to_revenue,daily_ev_multiple,0.0042,0.4915,-18.4267


## Feature Robustness Across Horizons

A feature is more interesting when it stays useful across multiple horizons, not just one window. This table ranks features by average rank IC across 20, 60, and 120 trading days.


In [9]:
robustness = (
    results_df.groupby(["feature", "variant", "family", "expected_direction"])
    .agg(
        horizons=("horizon", "nunique"),
        avg_rank_ic=("mean_daily_rank_ic", "mean"),
        min_rank_ic=("mean_daily_rank_ic", "min"),
        avg_spread_bps=("spread_bps", "mean"),
        positive_horizons=("mean_daily_rank_ic", lambda x: int((x > 0).sum())),
    )
    .reset_index()
    .sort_values(["positive_horizons", "avg_rank_ic", "avg_spread_bps"], ascending=[False, False, False])
)
display(robustness.head(25))
display(robustness.tail(15))


,feature,variant,family,expected_direction,horizons,avg_rank_ic,min_rank_ic,avg_spread_bps,positive_horizons
54,static_dividend_yield,static_vendor_ratio,dividend_yield,higher_is_better,3,0.0806,0.0403,144.5932,3
7,asof_quick_ratio,asof_quality_control,quick_ratio,higher_is_better,3,0.0676,0.0448,55.7463,3
0,asof_current_ratio,asof_quality_control,current_ratio,higher_is_better,3,0.0624,0.0374,72.7783,3
52,daily_total_debt_to_mcap,daily_market_cap_yield,total_debt,higher_is_better,3,0.0379,0.0251,476.2165,3
42,daily_mcap_to_total_debt,daily_market_cap_multiple,total_debt,lower_is_better,3,0.0332,0.0195,468.7306,3
43,daily_net_debt_to_mcap,daily_market_cap_yield,net_debt,higher_is_better,3,0.0096,0.0020,253.7354,3
13,daily_cash_to_mcap,daily_market_cap_yield,cash,higher_is_better,3,0.0082,0.0012,373.9627,3
30,daily_mcap_to_cash,daily_market_cap_multiple,cash,lower_is_better,3,0.0082,0.0012,373.9627,3
8,asof_return_on_assets,asof_quality_control,return_on_assets,higher_is_better,3,0.0126,-0.0018,-407.9390,2
57,static_ev_to_ocf,static_vendor_ratio,ev_to_ocf,lower_is_better,3,0.0093,-0.0077,-136.2550,2


,feature,variant,family,expected_direction,horizons,avg_rank_ic,min_rank_ic,avg_spread_bps,positive_horizons
51,daily_tangible_book_to_mcap,daily_market_cap_yield,tangible_book,higher_is_better,3,-0.0342,-0.0541,-401.2787,0
17,daily_ebitda_to_mcap,daily_market_cap_yield,ebitda,higher_is_better,3,-0.0342,-0.0576,-863.2174,0
6,asof_operating_profit_margin,asof_quality_control,operating_profit_margin,higher_is_better,3,-0.0348,-0.0509,-665.9291,0
45,daily_operating_cash_flow_to_ev,daily_ev_yield,ev_operating_cash_flow,higher_is_better,3,-0.0351,-0.0603,-541.1681,0
16,daily_ebitda_to_ev,daily_ev_yield,ev_ebitda,higher_is_better,3,-0.0356,-0.0618,-853.1398,0
33,daily_mcap_to_ebitda,daily_market_cap_multiple,ebitda,lower_is_better,3,-0.0363,-0.0504,-711.2004,0
14,daily_ebit_to_ev,daily_ev_yield,ev_ebit,higher_is_better,3,-0.0369,-0.0702,-751.1058,0
19,daily_ev_to_ebitda,daily_ev_multiple,ev_ebitda,lower_is_better,3,-0.0377,-0.0545,-688.2442,0
47,daily_operating_income_to_ev,daily_ev_yield,ev_operating_income,higher_is_better,3,-0.0456,-0.0730,-728.2329,0
28,daily_gross_profit_to_mcap,daily_market_cap_yield,gross_profit,higher_is_better,3,-0.0731,-0.1020,-915.9184,0


## Runtime Scaling Estimate

The configured run executes first. This section estimates how runtime could scale when expanding to all MAG7 symbols and a longer history. It is a projection from observed panel size, not a guarantee; if the projected runtime gets too high, optimize before scaling.

In [10]:
smoke_rows = len(panel)
smoke_symbols = len(ANALYSIS_SYMBOLS)
smoke_features = len(feature_cols)
smoke_start = pd.Timestamp(START_DATE)
smoke_end = pd.Timestamp(panel["date"].max())
smoke_days = max((smoke_end - smoke_start).days, 1)

full_start = pd.Timestamp(FULL_RUN_BASELINE_START_DATE)
full_days = max((smoke_end - full_start).days, 1)
full_symbol_factor = len(FULL_RUN_BASELINE_SYMBOLS) / max(smoke_symbols, 1)
full_history_factor = full_days / smoke_days
projected_full_rows = int(smoke_rows * full_symbol_factor * full_history_factor)

observed_total_seconds = run_timings.get("panel_build_seconds", 0.0) + run_timings.get("evaluation_seconds", 0.0)
projected_full_seconds = observed_total_seconds * full_symbol_factor * full_history_factor
projection = pd.DataFrame([
    {
        "run": "current_run",
        "symbols": smoke_symbols,
        "start_date": START_DATE,
        "rows": smoke_rows,
        "features": smoke_features,
        "panel_build_seconds": run_timings.get("panel_build_seconds", np.nan),
        "evaluation_seconds": run_timings.get("evaluation_seconds", np.nan),
        "total_seconds": observed_total_seconds,
    },
    {
        "run": "projected_mag7_since_2018",
        "symbols": len(FULL_RUN_BASELINE_SYMBOLS),
        "start_date": FULL_RUN_BASELINE_START_DATE,
        "rows": projected_full_rows,
        "features": smoke_features,
        "panel_build_seconds": run_timings.get("panel_build_seconds", np.nan) * full_symbol_factor * full_history_factor,
        "evaluation_seconds": run_timings.get("evaluation_seconds", np.nan) * full_symbol_factor * full_history_factor,
        "total_seconds": projected_full_seconds,
    },
])
projection["total_minutes"] = projection["total_seconds"] / 60.0
display(projection)

if projected_full_seconds > 120:
    display(Markdown("> Projected full run is above two minutes. Optimize before scaling: cache warehouse reads, reduce displayed intermediate tables, and vectorize top/bottom spread selection."))
else:
    display(Markdown("> Projected full MAG7 run is acceptable for a notebook. Larger universes should still move feature construction/evaluation into reusable Python helpers."))


,run,symbols,start_date,rows,features,panel_build_seconds,evaluation_seconds,total_seconds,total_minutes
0,current_run,7,2018-01-01,14910,59,0.2843,0.9140,1.1983,0.0200
1,projected_mag7_since_2018,7,2018-01-01,14910,59,0.2843,0.9140,1.1983,0.0200


> Projected full MAG7 run is acceptable for a notebook. Larger universes should still move feature construction/evaluation into reusable Python helpers.

## Written Analysis

This cell is filled after the notebook is executed, using the computed tables above.


In [11]:
def _fmt_seconds(value: float) -> str:
    if pd.isna(value):
        return "nan"
    return f"{value:,.2f}s"

current_total = run_timings.get("panel_build_seconds", 0.0) + run_timings.get("evaluation_seconds", 0.0)
projection_row = projection.loc[projection["run"].eq("projected_mag7_since_2018")].iloc[0]
optimized_subset = runtime_benchmark.loc[runtime_benchmark["method"].eq("array_evaluator_correctness_subset")].iloc[0]
baseline_subset = runtime_benchmark.loc[runtime_benchmark["method"].eq("baseline_loop_correctness_subset")].iloc[0]
full_optimized = runtime_benchmark.loc[runtime_benchmark["method"].eq("array_evaluator_full")].iloc[0]

winner_daily = winner_df["variant"].str.startswith("daily_").sum()
winner_static = winner_df["variant"].eq("static_vendor_ratio").sum()
winner_total = len(winner_df)
variant_120 = summary_by_variant.loc[summary_by_variant["horizon"].eq(120)].sort_values("median_rank_ic", ascending=False).head(3)
variant_120_lines = [
    f"- `{row.variant}`: median rank IC {row.median_rank_ic:.4f}, median spread {row.median_spread_bps:,.1f} bps"
    for row in variant_120.itertuples(index=False)
]

analysis_md = "\n".join([
    "### MAG7 Full-History Analysis From This Run",
    "",
    f"The notebook completed end to end on {panel['symbol'].nunique()} symbols, {len(panel):,} symbol-days, {len(feature_cols)} engineered features, and {len(HORIZONS)} forward-return horizons from {panel['date'].min().date()} through {panel['date'].max().date()}.",
    f"Panel construction took {_fmt_seconds(run_timings.get('panel_build_seconds', np.nan))}; array full evaluation took {_fmt_seconds(run_timings.get('evaluation_seconds', np.nan))}; total measured core runtime was {_fmt_seconds(current_total)}.",
    "",
    "#### Correctness Checks",
    "- Daily historical market cap is present for every MAG7 symbol and behaves like trading-day data.",
    "- Daily enterprise value was reconstructed from historical market cap plus lagged debt minus lagged cash.",
    "- No `historical_enterprise_value`, snapshot share-statistics, or direct FMP API calls were used.",
    f"- Baseline-loop and array evaluators matched on the benchmark subset; max mean-rank-IC difference was {max_metric_diff:.12f}; max spread difference was {max_spread_diff:.12f}.",
    "",
    "#### Runtime Checks",
    f"- Source-frame alignment across symbols took {_fmt_seconds(run_timings.get('source_align_seconds', np.nan))}; this is the part that replaces per-column as-of reprojection.",
    f"- Baseline subset: {int(baseline_subset['features'])} features x {int(baseline_subset["horizons"])} horizons in {_fmt_seconds(baseline_subset['seconds'])}.",
    f"- Array evaluator subset: same subset in {_fmt_seconds(optimized_subset['seconds'])}.",
    f"- Array evaluator full run: {int(full_optimized['features'])} features x {len(HORIZONS)} horizons in {_fmt_seconds(full_optimized['seconds'])}.",
    f"- Current MAG7 since 2018 projected/runtime estimate: {_fmt_seconds(float(projection_row['total_seconds']))} ({projection_row['total_minutes']:.2f} minutes).",
    "",
    "#### Validation Method Tradeoff",
    f"- Fast pooled validation ran in {_fmt_seconds(float(validation_timing.loc[validation_timing['method'].eq('fast_pooled_pearson'), 'seconds'].iloc[0]))}; it is appropriate for large-universe first-pass screening but ignores daily cross-sectional structure.",
    f"- Rank-IC-only validation ran in {_fmt_seconds(float(validation_timing.loc[validation_timing['method'].eq('rank_ic_only'), 'seconds'].iloc[0]))}; it matched the full evaluator rank IC with max error {method_agreement_error:.12f} and is the preferred scalable first-pass metric.",
    f"- Full array validation ran in {_fmt_seconds(float(validation_timing.loc[validation_timing['method'].eq('full_array_evaluator'), 'seconds'].iloc[0]))}; use it when spread diagnostics matter.",
    "",
    "#### Daily-Adjusted Versus Static/As-Of Result",
    f"Across the comparable valuation families and horizons, daily-adjusted variants won {winner_daily} of {winner_total} family-horizon comparisons by mean daily rank IC; static vendor ratios won {winner_static}.",
    "The result is not a broad win for daily-adjusted valuation features in this MAG7-only test. Daily market-cap and reconstructed-EV features occasionally beat their static counterpart for specific families, but the aggregate variant table is mostly flat to negative, especially at 60d and 120d.",
    "At the variant level for 120d returns:",
    *variant_120_lines,
    "",
    "#### Interpretation",
    "This does not disprove daily-adjusted fundamentals; it says MAG7 is too narrow and too idiosyncratic for a promotion decision. The implementation works and runs fast enough, but the signal evidence is weak in this universe. The next meaningful test should use a larger liquid equity universe with sector/industry-neutral ranks and liquidity filters.",
])

display(Markdown(analysis_md))


### MAG7 Full-History Analysis From This Run

The notebook completed end to end on 7 symbols, 14,910 symbol-days, 59 engineered features, and 3 forward-return horizons from 2018-01-02 through 2026-06-24.
Panel construction took 0.28s; array full evaluation took 0.91s; total measured core runtime was 1.20s.

#### Correctness Checks
- Daily historical market cap is present for every MAG7 symbol and behaves like trading-day data.
- Daily enterprise value was reconstructed from historical market cap plus lagged debt minus lagged cash.
- No `historical_enterprise_value`, snapshot share-statistics, or direct FMP API calls were used.
- Baseline-loop and array evaluators matched on the benchmark subset; max mean-rank-IC difference was 0.000000000540; max spread difference was 0.000000000301.

#### Runtime Checks
- Source-frame alignment across symbols took 0.06s; this is the part that replaces per-column as-of reprojection.
- Baseline subset: 6 features x 1 horizons in 7.72s.
- Array evaluator subset: same subset in 0.07s.
- Array evaluator full run: 59 features x 3 horizons in 0.91s.
- Current MAG7 since 2018 projected/runtime estimate: 1.20s (0.02 minutes).

#### Validation Method Tradeoff
- Fast pooled validation ran in 0.08s; it is appropriate for large-universe first-pass screening but ignores daily cross-sectional structure.
- Rank-IC-only validation ran in 0.53s; it matched the full evaluator rank IC with max error 0.000000358249 and is the preferred scalable first-pass metric.
- Full array validation ran in 0.91s; use it when spread diagnostics matter.

#### Daily-Adjusted Versus Static/As-Of Result
Across the comparable valuation families and horizons, daily-adjusted variants won 19 of 27 family-horizon comparisons by mean daily rank IC; static vendor ratios won 8.
The result is not a broad win for daily-adjusted valuation features in this MAG7-only test. Daily market-cap and reconstructed-EV features occasionally beat their static counterpart for specific families, but the aggregate variant table is mostly flat to negative, especially at 60d and 120d.
At the variant level for 120d returns:
- `asof_quality_control`: median rank IC -0.0145, median spread -978.5 bps
- `static_vendor_ratio`: median rank IC -0.0161, median spread 75.0 bps
- `daily_market_cap_multiple`: median rank IC -0.0314, median spread -662.8 bps

#### Interpretation
This does not disprove daily-adjusted fundamentals; it says MAG7 is too narrow and too idiosyncratic for a promotion decision. The implementation works and runs fast enough, but the signal evidence is weak in this universe. The next meaningful test should use a larger liquid equity universe with sector/industry-neutral ranks and liquidity filters.